# TEST — LOAD CSV

Exploratory notebook to understand how `LOAD CSV` works in TuringDB v1.23.

## Key finding

`LOAD CSV` in TuringDB **is NOT the same as `LOAD JSONL`**. It does not create a self-contained graph from a file. It is a row-based reader inspired by Neo4j's `LOAD CSV`, used inside a write transaction.

### Correct pattern

```python
client.query('create graph my_graph')
client.set_graph('my_graph')
client.new_change()                          # open write transaction
client.query("LOAD CSV 'file.csv' WITH HEADERS AS row CREATE (n:Label)")
client.query('CHANGE SUBMIT')               # commit
client.checkout()                           # back to main branch
```

### What works
| Feature | Status |
|---|---|
| `RETURN row.ColumnName` | ✅ read CSV values |
| `CREATE (n:Label)` | ✅ creates one node per row |
| `WITH HEADERS AS row` | ✅ gives named column access |

### What does NOT work
| Feature | Status |
|---|---|
| `CREATE (n:Label {prop: row.Col})` | ❌ internal assertion error |
| `SET n.prop = row.Col` | ❌ type `Invalid` incompatible |
| `WHERE row.Col = 'value'` | ❌ parse error |
| `MERGE` | ❌ not implemented |
| `LOAD CSV 'file' AS graph_name` (LOAD JSONL-style) | ❌ parse error |

**Conclusion:** LOAD CSV cannot transfer CSV values into node/edge properties in v1.23. For data loading, continue using **LOAD JSONL**.

## Setup

In [1]:
import os
import shutil
from pathlib import Path
import turingdb
from IPython.display import display

client = turingdb.TuringDB()
print('TuringDB connected')

TuringDB connected


In [2]:
# TuringDB data directory (paths in LOAD CSV are relative to this)
DATA_DIR = Path.home() / '.turing' / 'data'
print(f'TuringDB data dir: {DATA_DIR}')
print(f'Exists: {DATA_DIR.exists()}')

TuringDB data dir: /home/ubuntu/.turing/data
Exists: True


In [3]:
# Copy the CSVs we want to test into the TuringDB data directory
REPO_DATA = Path(os.getcwd()) / 'data'

csv_files = {
    'healthcare_dataset.csv':          REPO_DATA / 'healthcare_dataset'     / 'healthcare_dataset.csv',
    'tfl_tube.csv':                    REPO_DATA / 'london_transport_TfL'   / 'TfL_london_transport_tube.csv',
    'tfl_stations.csv':                REPO_DATA / 'london_transport_TfL'   / 'TfL_london_transport_tube_stations.csv',
}

for dest_name, src_path in csv_files.items():
    dest = DATA_DIR / dest_name
    shutil.copy(src_path, dest)
    print(f'  Copied {src_path.name} -> {dest}')

  Copied healthcare_dataset.csv -> /home/ubuntu/.turing/data/healthcare_dataset.csv
  Copied TfL_london_transport_tube.csv -> /home/ubuntu/.turing/data/tfl_tube.csv
  Copied TfL_london_transport_tube_stations.csv -> /home/ubuntu/.turing/data/tfl_stations.csv


---
## What works: RETURN row values

`LOAD CSV 'file' WITH HEADERS AS row RETURN row.ColumnName` works — useful for inspecting a CSV without importing it.

In [4]:
# Inspect healthcare CSV — no graph context needed
result = client.query("LOAD CSV 'healthcare_dataset.csv' WITH HEADERS AS row RETURN row.Name, row.Age, row.Gender LIMIT 5")
display(result)

,row.Name,row.Age,row.Gender
0,Bobby JacksOn,30,Male
1,LesLie TErRy,62,Male
2,DaNnY sMitH,76,Female
3,andrEw waTtS,28,Female
4,adrIENNE bEll,43,Female


In [5]:
# Works on edge-list CSVs too
result = client.query("LOAD CSV 'tfl_tube.csv' WITH HEADERS AS row RETURN row.Line, row.From_Station, row.To_Station LIMIT 5")
display(result)

,row.Line,row.From_Station,row.To_Station
0,Bakerloo,Elephant & Castle,Lambeth North
1,Bakerloo,Lambeth North,Waterloo
2,Bakerloo,Waterloo,Embankment
3,Bakerloo,Embankment,Charing Cross
4,Bakerloo,Charing Cross,Piccadilly Circus


---
## What works: CREATE labeled nodes (no properties)

`LOAD CSV ... CREATE (n:Label)` creates one node per CSV row inside a write transaction.
The nodes have a label but **no properties** — values from `row` cannot be transferred.

In [6]:
%%time
# Create a graph and load healthcare CSV (55 500 rows)
# test_healthcare_csv was already created in a previous run — reuse it, clear it first
client.set_graph('test_healthcare_csv')
client.new_change()
client.query('MATCH (n) DETACH DELETE n')
client.query('CHANGE SUBMIT')
client.checkout()

client.new_change()
client.query("LOAD CSV 'healthcare_dataset.csv' WITH HEADERS AS row CREATE (n:Patient)")
client.query('CHANGE SUBMIT')
client.checkout()
print('Done')

Done
CPU times: user 6.68 ms, sys: 0 ns, total: 6.68 ms
Wall time: 97.5 ms


In [7]:
# 55 500 Patient nodes created — one per CSV row
print('Node count:')
display(client.query('MATCH (n) RETURN count(n)'))

# Label is correct
print('\nLabels:')
display(client.query('CALL db.labels()'))

# But NO properties — row values were not transferred
print('\nProperty types (expected: empty):')
display(client.query('CALL db.propertyTypes()'))

Node count:


,count(n)
0,55500



Labels:


,id,label
0,0,Patient
1,1,Station



Property types (expected: empty):


,id,propertyType,valueType


In [8]:
%%time

# CALL propertyTypes() - returns a column of all the different node and edge properties and their types in the database
command = """
CALL db.propertyTypes()
"""
df_propertyTypes = client.query(command)
if df_propertyTypes.empty:
    print("No result found")
else:
    display(df_propertyTypes)

No result found
CPU times: user 0 ns, sys: 1.67 ms, total: 1.67 ms
Wall time: 1.34 ms


In [9]:
# Get node properties
nodes_properties = df_propertyTypes["propertyType"].values.tolist()
print(f"Node properties: {nodes_properties}")

Node properties: []


In [10]:
%%time

# CALL labels () - returns a column of all the different node labels
command = """
CALL db.labels()
"""
df_labels = client.query(command)
if df_labels.empty:
    print("No result found")
else:
    display(df_labels)

,id,label
0,0,Patient
1,1,Station


CPU times: user 1.21 ms, sys: 3.91 ms, total: 5.12 ms
Wall time: 4.72 ms


In [11]:
%%time

# CALL edgeTypes() - returns a column of all the different edge types (edge equivalent of node labels)
command = """
CALL db.edgeTypes()
"""
df_edgeTypes = client.query(command)
if df_edgeTypes.empty:
    print("No result found")
else:
    display(df_edgeTypes)

No result found
CPU times: user 1.91 ms, sys: 0 ns, total: 1.91 ms
Wall time: 1.65 ms


In [13]:
client.query("MATCH (n)-[e]->(m) RETURN count(e)")

,count(e)
0,0


---
## Limitations

### CREATE with properties — fails (internal assertion error)

In [8]:
try:
    client.new_change()
    client.query("LOAD CSV 'healthcare_dataset.csv' WITH HEADERS AS row CREATE (n:Patient {name: row.Name})")
    client.query('CHANGE SUBMIT')
    client.checkout()
except Exception as e:
    client.checkout()
    print(f'Expected error: {e}')

Expected error: EXEC_ERROR: Unexpected exception: Internal Error: The assertion 'casted' failed at /home/runner/work/turingdb/turingdb/query/pipeline/processors/WriteProcessor.cpp:83
Could not get constant property value.



### SET n.prop = row.col — fails (type `Invalid` incompatible)

In [9]:
# SET n.prop = row.col is refused at analyse time:
# "Cannot evaluate property: types 'String' and 'Invalid' are incompatible"
# row.Col values have type 'Invalid' in the LOAD CSV context — they cannot be assigned to properties.
try:
    client.new_change()
    client.query("LOAD CSV 'tfl_tube.csv' WITH HEADERS AS row CREATE (n:Station) SET n.type = row.Line")
    client.query('CHANGE SUBMIT')
    client.checkout()
except Exception as e:
    client.checkout()
    print(f'Expected error: {e}')

Expected error: ANALYZE_ERROR: -------* Query error
     1 | LOAD CSV 'tfl_tube.csv' WITH HEADERS AS row CREATE (n:Station) SET n.type = row.Line
       |                                                                    ^^^^^^
-------* Property type 'type' is invalid


### WHERE row.Col = 'value' — fails (parse error)

In [10]:
try:
    client.query("LOAD CSV 'tfl_tube.csv' WITH HEADERS AS row WHERE row.Line = 'Bakerloo' RETURN row.From_Station LIMIT 3")
except Exception as e:
    print(f'Expected error: {e}')

Expected error: PARSE_ERROR: -------* Cypher parser
     1 | LOAD CSV 'tfl_tube.csv' WITH HEADERS AS row WHERE row.Line = 'Bakerloo' RETURN row.From_Station LIMIT 3
       |                                             ^^^^^
-------* syntax error, unexpected WHERE


---
## Conclusion

`LOAD CSV` in TuringDB v1.23 is a **read-only CSV inspector + skeleton node creator**:

- ✅ `RETURN row.Col` — read and preview CSV data (no graph write needed)
- ✅ `CREATE (n:Label)` inside a write transaction — bulk-creates labeled empty nodes (one per row)
- ❌ Properties from `row` cannot be set on nodes/edges (type system limitation: `row.Col` has type `Invalid`)
- ❌ `WHERE`, `MERGE`, inline property maps `{prop: row.Col}` are not supported

**Practical impact:** LOAD CSV cannot replace LOAD JSONL for data ingestion. All existing example notebooks should continue using LOAD JSONL. LOAD CSV is only useful for quick inspection of CSV contents.